# FireRisk AI — Weather-Based Forest Fire Occurrence Prediction

**Statistics Mini Project — Supervised Learning using Python**

### Objective
Estimate whether a day is associated with **fire** or **not fire** using weather conditions and Fire Weather Index (FWI) variables.

> This is an educational risk-classification prototype, not an operational fire-warning system.

**Dataset:** Algerian Forest Fires Dataset, Bejaia Region (122 daily observations), originally part of the UCI Algerian Forest Fires dataset. The original UCI collection contains 244 observations from two Algerian regions.  
Source: https://archive.ics.uci.edu/dataset/547/algerian+forest+fire+dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("fire_risk_dataset_bejaia.csv")
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nClass distribution:")
print(df["Classes"].value_counts())


## 1. Problem Statement

Forest fires are influenced by meteorological and fire-weather conditions. The aim is to build a supervised classification model that estimates whether a day is associated with a **fire** or **not fire** based on observed environmental conditions.

The project demonstrates data preprocessing, descriptive statistics, exploratory data analysis, statistical testing, supervised learning and model evaluation.


In [ ]:
df["target"] = (df["Classes"].str.strip().str.lower() == "fire").astype(int)

features = ["Temperature","RH","Ws","Rain","FFMC","DMC","DC","ISI","BUI","FWI"]
X = df[features]
y = df["target"]

df[features].describe().T


In [ ]:
# Class distribution
df["Classes"].value_counts().plot(kind="bar")
plt.title("Fire vs Not Fire")
plt.xlabel("Class")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()


In [ ]:
# Fire occurrence rate by month
df.groupby("month")["target"].mean().plot(kind="bar")
plt.title("Fire Occurrence Rate by Month")
plt.xlabel("Month")
plt.ylabel("Fire occurrence rate")
plt.tight_layout()
plt.show()


In [ ]:
# Compare means
df.groupby("Classes")[features].mean().T


In [ ]:
# Correlation with fire target
df[features + ["target"]].corr()["target"].drop("target").sort_values(key=abs, ascending=False)


In [ ]:
# Two-sample Welch t-tests
from scipy.stats import ttest_ind

test_results = []
for col in features:
    fire = df.loc[df["target"] == 1, col]
    no_fire = df.loc[df["target"] == 0, col]
    t_stat, p_value = ttest_ind(fire, no_fire, equal_var=False)
    test_results.append([col, t_stat, p_value])

ttest_df = pd.DataFrame(test_results, columns=["Variable","t_statistic","p_value"])
ttest_df.sort_values("p_value")


### Interpretation of the statistical tests

A small p-value indicates evidence that the mean of that variable differs between fire and not-fire observations in this sample. Statistical significance does **not** prove that a variable causes fires; it only indicates an association in the observed data.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    )
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]
    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred),
        recall_score(y_test, pred),
        f1_score(y_test, pred),
        roc_auc_score(y_test, prob)
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model","Accuracy","Precision","Recall","F1","ROC-AUC"]
)
results_df


In [ ]:
# Confusion matrix for Random Forest
rf = models["Random Forest"]
pred = rf.predict(X_test)
cm = confusion_matrix(y_test, pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()


In [ ]:
# Random Forest feature importance
rf_importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
rf_importance.plot(kind="bar")
plt.title("Random Forest Feature Importance")
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()


## 2. Model Evaluation

The models are evaluated using accuracy, precision, recall, F1-score and ROC-AUC.

For a fire-risk application, recall is important because missing a fire-risk day (false negative) can be more concerning than generating an additional false alarm. However, the model should not be treated as an operational warning system.


## 3. Conclusion

The analysis shows clear differences in several environmental and Fire Weather Index variables between fire and not-fire observations. In this sample, higher temperature and fire-weather indicators are associated with fire observations, while higher humidity and rainfall are generally associated with non-fire observations.

Both Logistic Regression and Random Forest provide strong classification performance on the fixed train/test split. Because the dataset is small and covers only one region and a limited period, these results should not be interpreted as general performance for India or for future fire seasons.

### Limitations
1. The working dataset contains 122 observations from the Bejaia region.
2. The observations are from 2012 and cover June–September.
3. Satellite, land-use, vegetation, population and industrial variables are not included.
4. The model estimates statistical association, not causation.
5. A larger multi-year, multi-region dataset is required for a deployable fire-risk system.
